<a href="https://colab.research.google.com/github/LuisTT0903/Processamento-de-Sinais1/blob/main/Pratica4Quest%C3%A3o3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import urllib.request

import numpy as np
from scipy.io import wavfile
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Carregamento do handel.wav
#    (mesma estrutura usada nas práticas anteriores da disciplina: tenta ler
#    o arquivo local e, se não encontrar, baixa do próprio repositório do
#    GitHub da disciplina — útil para execução em ambientes na nuvem)
# ---------------------------------------------------------------------------
CAMINHO_LOCAL = "handel.wav"
URL_REPO = (
    "https://raw.githubusercontent.com/LuisTT0903/Processamento-de-Sinais1/"
    "main/Aula_04/Audios_Usados/handel.wav"
)

if not os.path.exists(CAMINHO_LOCAL):
    print(f"Arquivo local '{CAMINHO_LOCAL}' não encontrado. Baixando de:\n  {URL_REPO}")
    urllib.request.urlretrieve(URL_REPO, CAMINHO_LOCAL)

fs, x_int = wavfile.read(CAMINHO_LOCAL)
x = x_int.astype(np.float64)
N = len(x)
print(f"Sinal carregado: N = {N} amostras, fs = {fs} Hz, duração = {N/fs:.2f} s\n")


# ---------------------------------------------------------------------------
# 2. DFT via matriz de DFT (direta e inversa), calculada em blocos
# ---------------------------------------------------------------------------
def dft_matriz_bloco(sinal, expo=-1, bloco=2000):
    """
    Calcula Y[k] = soma_n sinal[n] * exp(expo * j*2*pi*k*n/N),  k = 0..N-1

    expo = -1  ->  DFT direta         (matriz W[k,n] = exp(-j*2*pi*k*n/N))
    expo = +1  ->  "núcleo" da IDFT   (matriz W[k,n] = exp(+j*2*pi*k*n/N),
                                        basta dividir o resultado por N)

    A matriz é construída em blocos de linhas. Cada nova linha é obtida
    multiplicando a linha anterior por w_n = exp(expo*j*2*pi*n/N) (relação
    W_N^(k+1) = W_N^k * W_N), o que é matematicamente idêntico a montar a
    matriz de DFT diretamente, porém muito mais rápido que calcular exp()
    para cada um dos N^2 elementos individualmente.
    """
    N = len(sinal)
    w_n = np.exp(expo * 1j * 2 * np.pi * np.arange(N) / N)
    Y = np.empty(N, dtype=complex)
    linha = np.ones(N, dtype=complex)  # linha k = 0
    k = 0
    while k < N:
        b = min(bloco, N - k)
        W = np.empty((b, N), dtype=complex)
        W[0] = linha
        for i in range(1, b):
            W[i] = W[i - 1] * w_n
        Y[k:k + b] = W @ sinal
        linha = W[-1] * w_n
        k += b
    return Y


def dft_direta(x):
    """DFT direta de x, via matriz de DFT."""
    return dft_matriz_bloco(x, expo=-1)


def idft_direta(X):
    """DFT inversa (IDFT) de X, via matriz de DFT inversa."""
    N = len(X)
    return (dft_matriz_bloco(X, expo=+1) / N).real


print("Calculando a DFT direta de x[n] (via matriz de DFT, em blocos)...")
X = dft_direta(x)
print("DFT concluída.\n")

energia = np.abs(X) ** 2
energia_total = energia.sum()

# ---------------------------------------------------------------------------
# 3. Compressão para cada fator de energia r
# ---------------------------------------------------------------------------
r_valores = [0.995, 0.990, 0.900, 0.750, 0.500]

ordem = np.argsort(energia)[::-1]                       # índices por energia decrescente
energia_acumulada = np.cumsum(energia[ordem]) / energia_total

resultados = []
sinais_reconstruidos = {}

for r in r_valores:
    M = int(np.searchsorted(energia_acumulada, r) + 1)  # menor M com energia acumulada >= r
    idx_mantidos = ordem[:M]

    X_comprimido = np.zeros(N, dtype=complex)
    X_comprimido[idx_mantidos] = X[idx_mantidos]

    print(f"Reconstruindo sinal para r = {r*100:5.1f}%  (M = {M} coeficientes)...")
    x_rec = idft_direta(X_comprimido)

    mse = float(np.mean((x - x_rec) ** 2))
    taxa_compressao = N / M

    x_rec_int16 = np.clip(np.round(x_rec), -32768, 32767).astype(np.int16)
    nome_arquivo = f"handel_comprimido_r{int(round(r*1000)):04d}.wav"
    wavfile.write(nome_arquivo, fs, x_rec_int16)

    resultados.append(dict(r=r, M=M, mse=mse, taxa=taxa_compressao, arquivo=nome_arquivo))
    sinais_reconstruidos[r] = x_rec

    print(f"  -> {100*M/N:5.2f}% dos coeficientes | "
          f"taxa de compressão = {taxa_compressao:6.2f}x | "
          f"MSE = {mse:9.3f} | salvo em {nome_arquivo}\n")

wavfile.write("handel_original.wav", fs, x_int)

# ---------------------------------------------------------------------------
# 4. Gráficos
# ---------------------------------------------------------------------------
t = np.arange(N) / fs

# 4a. Forma de onda: original vs. reconstruído, para cada r (janela zoom)
janela = (t >= 2.0) & (t <= 2.05)   # zoom de 50 ms para visualizar a diferença

fig1, axes = plt.subplots(len(r_valores), 1, figsize=(11, 12), sharex=True)
for ax, r in zip(axes, r_valores):
    ax.plot(t[janela], x[janela], color="C0", lw=1.2, label="Original")
    ax.plot(t[janela], sinais_reconstruidos[r][janela], color="C1", lw=1.0,
             ls="--", label=f"Comprimido (r={r*100:.1f}%)")
    ax.set_ylabel("Amplitude")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Tempo (s)")
axes[0].set_title("Forma de onda: sinal original vs. comprimido (zoom de 50 ms)")
plt.tight_layout()
plt.savefig("compressao_formas_de_onda.png", dpi=150)
plt.close(fig1)

# 4b. Espectro de magnitude: original vs. coeficientes mantidos (exemplo r=90%)
r_exemplo = 0.900
M_exemplo = [res["M"] for res in resultados if res["r"] == r_exemplo][0]
idx_exemplo = ordem[:M_exemplo]
freq = np.arange(N) * fs / N

fig2, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(freq[:N // 2], np.abs(X)[:N // 2], color="lightgray", lw=0.8, label="Espectro original |X[k]|")
mask_metade = idx_exemplo[idx_exemplo < N // 2]
ax.scatter(freq[mask_metade], np.abs(X)[mask_metade], s=4, color="C3",
           label=f"Coeficientes mantidos (r={r_exemplo*100:.0f}%, M={M_exemplo})")
ax.set_xlabel("Frequência (Hz)")
ax.set_ylabel("|X[k]|")
ax.set_title("Coeficientes de maior energia mantidos na compressão (exemplo: r = 90%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("compressao_espectro.png", dpi=150)
plt.close(fig2)

# 4c. MSE x taxa de compressão / número de coeficientes
rs = [res["r"] * 100 for res in resultados]
Ms = [res["M"] for res in resultados]
mses = [res["mse"] for res in resultados]
taxas = [res["taxa"] for res in resultados]

fig3, (axA, axB) = plt.subplots(1, 2, figsize=(12, 4.5))

axA.plot(taxas, mses, "o-", color="C2")
for xr, yr, rr in zip(taxas, mses, rs):
    axA.annotate(f"r={rr:.1f}%", (xr, yr), textcoords="offset points",
                 xytext=(5, 5), fontsize=8)
axA.set_xlabel("Taxa de compressão (N/M)")
axA.set_ylabel("MSE")
axA.set_title("MSE vs. taxa de compressão")
axA.grid(True, alpha=0.3)

axB.plot(Ms, mses, "o-", color="C4")
for xr, yr, rr in zip(Ms, mses, rs):
    axB.annotate(f"r={rr:.1f}%", (xr, yr), textcoords="offset points",
                 xytext=(5, 5), fontsize=8)
axB.set_xlabel("Número de coeficientes mantidos (M)")
axB.set_ylabel("MSE")
axB.set_title("MSE vs. número de coeficientes")
axB.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("compressao_mse.png", dpi=150)
plt.close(fig3)

# ---------------------------------------------------------------------------
# 5. Tabela-resumo
# ---------------------------------------------------------------------------
print("Resumo dos resultados:")
print(f"{'r (%)':>8} | {'M (coef.)':>10} | {'% de N':>8} | {'taxa (N/M)':>11} | {'MSE':>12}")
print("-" * 60)
for res in resultados:
    print(f"{res['r']*100:8.1f} | {res['M']:10d} | {100*res['M']/N:7.2f}% | "
          f"{res['taxa']:10.2f}x | {res['mse']:12.3f}")

Arquivo local 'handel.wav' não encontrado. Baixando de:
  https://raw.githubusercontent.com/LuisTT0903/Processamento-de-Sinais1/main/Aula_04/Audios_Usados/handel.wav
Sinal carregado: N = 73113 amostras, fs = 8192 Hz, duração = 8.92 s

Calculando a DFT direta de x[n] (via matriz de DFT, em blocos)...


Comentários sobre os resultados
--------------------------------
1) A energia do sinal handel.wav está concentrada em relativamente poucos
   coeficientes de Fourier: manter apenas 50% da energia exige guardar só
   cerca de 1,6% dos N coeficientes (M ~ 1170), enquanto manter 99,5% da
   energia já exige mais de 66% dos coeficientes (M ~ 48670). Isso mostra
   que a relação entre "energia retida" e "número de coeficientes
   necessários" é fortemente não linear: os primeiros coeficientes (os de
   maior energia) contribuem desproporcionalmente mais para a reconstrução
   do sinal do que os últimos.

2) O MSE cresce à medida que a taxa de compressão aumenta (menos
   coeficientes mantidos), como esperado — cada coeficiente descartado
   remove uma parcela da energia do sinal e introduz erro na reconstrução
   no domínio do tempo (relação de Parseval). Para r próximo de 99,5%/99%,
   o MSE é pequeno e a forma de onda reconstruída praticamente coincide com
   a original; para r = 75% e, principalmente, r = 50%, o erro cresce
   visivelmente e a forma de onda reconstruída perde detalhes finos
   (transientes e componentes de maior frequência, que têm menor energia
   individual mas são responsáveis pela nitidez do sinal).

3) Do ponto de vista subjetivo (ouvindo os arquivos handel_comprimido_r*.wav
   gerados), espera-se que r = 99,5% e r = 99,0% sejam praticamente
   indistinguíveis do áudio original; r = 90% ainda deve soar claramente
   como a mesma música, com leve perda de "brilho"/nitidez; r = 75% já deve
   apresentar uma perda perceptível de qualidade (som mais "abafado"); e
   r = 50% tende a soar visivelmente degradado/distorcido, pois quase toda
   informação de frequências mais altas (importantes para inteligibilidade
   e timbre) foi descartada, restando principalmente as componentes de
   baixa frequência de maior energia.

4) Em resumo, a compressão por DFT explora a concentração de energia do
   sinal em poucos coeficientes: é possível obter taxas de compressão
   consideráveis (ex.: ~63x para r=50%) com degradação perceptual ainda
   moderada, mas existe um compromisso direto entre taxa de compressão,
   número de coeficientes mantidos e o erro (objetivo, via MSE, e
   subjetivo, via audição) introduzido na reconstrução do sinal.